In [ ]:
import torch

inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89], # Your (x^1)
        [0.55, 0.87, 0.66], # Journey (x^1)
        [0.57, 0.85, 0.64], # Starts (x^1)
        [0.22, 0.58, 0.33], # With (x^1)
        [0.77, 0.25, 0.10], # One (x^1)
        [0.05, 0.80, 0.55], # Step (x^1)
    ]
)

In [ ]:
query = inputs[1] # The second input token serves as the query here
atten_scores_2 = torch.empty(inputs.shape[0])

for i, x_i in enumerate(inputs):
  atten_scores_2[i] = torch.dot(x_i, query)

print(atten_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [ ]:
#understanding the dot products

res = 0.
for idx, element in enumerate(inputs[0]):
  res += inputs[0][idx] * query[idx]
print(res)
print(torch.dot(inputs[0], query))


tensor(0.9544)
tensor(0.9544)


In [ ]:
#The main goal behind the normalization is to obtain attention weights that sum up to 1

atten_scores_2_temp = atten_scores_2 / atten_scores_2.sum()
print(f"Attention weights (Normalized)  {atten_scores_2_temp}")
print(f"Attention (Normalized) sum: {atten_scores_2_temp.sum()}")

Attention weights (Normalized)  tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Attention (Normalized) sum: 1.0000001192092896


In [ ]:
#More advisable to use softmax, better solution for managing extreme values and offers more favorable

def softmax(x): #numerical instability problems, such as overflow and underflow,
  return torch.exp(x) / torch.exp(x).sum(dim=0)

atten_scores_2_native = softmax(atten_scores_2)
print(f"Attention weights (Normalized)  {atten_scores_2_native}")
print(f"Attention (Normalized) sum: {atten_scores_2_native.sum()}")


Attention weights (Normalized)  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Attention (Normalized) sum: 1.0


In [ ]:
#use torch softmax for stability

atten_weights_2 = torch.softmax(atten_scores_2, dim=0)
print(f"Attention weights (Normalized)  {atten_weights_2}")
print(f"Attention (Normalized) sum: {atten_weights_2.sum()}")

Attention weights (Normalized)  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Attention (Normalized) sum: 1.0


In [ ]:
#  calculating the context vector z(2) by multiplying the
# embedded input tokens, x(i), with the corresponding attention weights and then sum
# ming the resulting vectors.


query = inputs[1] # 2nd input is the query
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
  context_vec_2 += atten_weights_2[i] * x_i

print(f"The context vector for given query : \n{context_vec_2}")



The context vector for given query : 
tensor([0.4419, 0.6515, 0.5683])


# 3.3.2 Computing attention weights for all input tokens

In [ ]:
atten_scores = torch.empty(6,6)

for i, x_i in enumerate(inputs):
  for j, x_j in enumerate(inputs):
    atten_scores[i,j] = torch.dot(x_i, x_j)
print(atten_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [ ]:
# instead of for loops (nested) which are slow, use the built-in
atten_scores = inputs @ inputs.T
print(atten_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [ ]:
atten_weights = torch.softmax(atten_scores, dim=-1) #it will normalize across the columns so that the values in each row (summing over the column dimension) sum up to 1.
print(atten_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [ ]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print(f"Row 2 sum: {row_2_sum}")
print(f"All rows sum: {atten_weights.sum(dim=-1)}")

Row 2 sum: 1.0
All rows sum: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [ ]:
#we use these attention weights to compute all context vectors via matrix multiplication:
all_context_vecs = atten_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [ ]:
print("Previous 2nd context vector:", context_vec_2) #sanity check of our formula with previous computed

Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


#3.4 Implementing self-attention with trainable weights
Also caled "scaled dot-product attention."

In [ ]:
#We will implement the self-attention mechanism step by step by introducing the three trainable weight matrices Wq, Wk, and Wv.
#These three matrices are used to project the embedded input tokens, x(i), into query, key, and value vectors, respectively,



In [ ]:
# we start here by computing only one context vector, z(2), for illustration purposes.

x_2 = inputs[1] #2nd input element
d_in = inputs.shape[1] #The input embedding size, d=3
d_out = 2 # The output_embedding size, d_out = 2

In [ ]:
# we initialize the three weight matrices Wq, Wk, and Wv
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad= False)
W_key   = torch.nn.Parameter(torch.rand(d_in,d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [ ]:

print(f"W_query : {W_query}")
print(f"W_key : {W_key}")
print(f"W_value : {W_value}")

print( f" x_2 : {x_2}" ) # x_2 shape is 1x3 and W shape is 3x2
#Calculate query, key, value
query_2 =  x_2 @ W_query
key_2 =   x_2 @ W_key
value_2 =  x_2 @ W_value
print(f"query for x_2 : {query_2}")
print(f"key for x_2 : {key_2}")
print(f"value for x_2 : {value_2}")


W_query : Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])
W_key : Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]])
W_value : Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])
 x_2 : tensor([0.5500, 0.8700, 0.6600])
query for x_2 : tensor([0.4306, 1.4551])
key for x_2 : tensor([0.4433, 1.1419])
value for x_2 : tensor([0.3951, 1.0037])


In [ ]:
print("Shapes involved : ")
print("inputs:  ", inputs.shape)
print("W_key:   ", W_key.shape)
print("W_value: ", W_value.shape)

Shapes involved : 
inputs:   torch.Size([6, 3])
W_key:    torch.Size([3, 2])
W_value:  torch.Size([3, 2])


In [ ]:
# even for calculating attention weights wrt query q(2) or x_2, all inputs are involved, hence calculate the keys, and values for each!
keys = inputs @ W_key
values = inputs @ W_value

print(f"All Keys Keys: {keys}")
print(f"All Value Keys: {values}")

print("\n\nShapes involved : ")
print("keys:  ", keys.shape)
print("values:   ", values.shape)


#As we can tell from the outputs, we successfully projected the six input tokens from a three-dimensional onto a two-dimensional embedding space:

All Keys Keys: tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]])
All Value Keys: tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]])


Shapes involved : 
keys:   torch.Size([6, 2])
values:    torch.Size([6, 2])


In [ ]:
# Now, computing the attention scores

# The new aspect here is that we are not directly computing the
# dot-product between the input elements but using the query and key obtained by transforming the inputs via the
# respective weight matrices.

#let's calculate the attention score w22

keys_2 = keys[1]
atten_score_22 = query_2.dot(keys_2)
print(atten_score_22)

tensor(1.8524)


In [ ]:
atten_scores_2 = query_2 @ keys.T
print(atten_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [ ]:
# scale the attention scores by dividing them by the square root of the embedding dimension of the keys
# Attention weights then applying the softmax on attention scores

d_k = keys.shape[-1]
atten_weights_2 = torch.softmax(atten_scores_2 / d_k**0.5, dim=-1)
# The scaling by the square root of the embedding dimension is the reason why this self-attention mechanism is also called scaled-dot product attention.
print(atten_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [ ]:
#Now, the final step is to compute the context vectors
context_vec_2 = atten_weights_2 @ values # single context vector, z(2).
print(context_vec_2)

tensor([0.3061, 0.8210])


# 3.4.2 Implementing a compact self-attention Python class

In [ ]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
  def __init__(self, d_in, d_out):
    super().__init__()
    self.W_query = nn.Parameter(torch.rand(d_in, d_out))
    self.W_key = nn.Parameter(torch.rand(d_in, d_out))
    self.W_value = nn.Parameter(torch.rand(d_in, d_out))

  def forward(self,x):
    keys = x @ self.W_key
    queries = x @ self.W_query
    values = x @ self.W_value

    attn_scores =  queries @ keys.T
    attn_weights = torch.softmax(
        attn_scores / keys.shape[-1] ** 0.5, dim=-1
    )
    context_vec  = attn_weights @ values
    return context_vec


In [ ]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [ ]:
# instead of manually implementing nn.Parameter(torch.rand(...)) is that nn.Linear
# has an optimized weight initialization scheme, contributing to more stable and
# effective model training.

import torch.nn as nn

class SelfAttention_v2(nn.Module):
  def __init__(self, d_in, d_out, qkv_bias=False):
    super().__init__()
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

  def forward(self,x):
    keys = self.W_key(x)
    queries =  self.W_query(x)
    values =  self.W_value(x)

    attn_scores =  queries @ keys.T
    attn_weights = torch.softmax(
        attn_scores / keys.shape[-1] ** 0.5, dim=-1
    )
    context_vec  = attn_weights @ values
    return context_vec


In [ ]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


In [ ]:
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


#3.5.1 Applying a causal attention mask

In [ ]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(
    attn_scores / keys.shape[-1] ** 0.5, dim=-1
)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [ ]:
#creating the mask with torch.tril

context_length = attn_scores.shape[0] #only having the n_rows because the attn wts is nxn matrix
mask_simple = torch.tril(torch.ones(context_length, context_length)) #creating a diagonal matrix of size nxn
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [ ]:
# 3rd step: Getting the masked attention weights. Simply multiply the mask with the attention weights
masked_simple = attn_weights * mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [ ]:
## 4th step: Normalize the masked attention weights so that each row sums to 1.
#
rows_sum = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / rows_sum
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [ ]:
# More efficient way
# 1. Calculate attention scores
# 2. Mask attention scores with -inf above diagonal
# 3. Normalize using the softmax to get the masked attention weights
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [ ]:
attn_weights = torch.softmax(
    masked / keys.shape[-1] ** 0.5, dim=-1
    )
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


# 3.5.2 Masking additional attention weights with dropout
- randomply selecting hidden layer units are ignored during training.
- to avoid overfitting
- only during training, and not inference

In [ ]:
# random dropout example
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6,6)
print(f"example matrix: \n{example}")
print(f"example matrix dropout: \n{dropout(example)}") #approximately half the values are zeroed out

example matrix: 
tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])
example matrix dropout: 
tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [ ]:
# 3rd step, applying the dropout directly to the attention scores (masked & normalized)
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


# 3.5.3 Implementing a compact causal attention class

In [ ]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) # two inputs with six tokens each; each token has embedding dimension 3.

torch.Size([2, 6, 3])


In [ ]:
class CausalAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
    super().__init__()
    self.d_out = d_out
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    self.dropout = nn.Dropout(dropout) # Added a dropoutlayer
    self.register_buffer(
        'mask',
        torch.triu(torch.ones(context_length, context_length),
                   diagonal=1)
    )

  def forward(self, x):
    b, num_tokens, d_in = x.shape # batch.shape as above
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    attn_scores = queries @ keys.transpose(1,2) # we transpose dimenstions 1 and 2, keeping the batch dimenstions at the first position (0)

    attn_scores.masked_fill(
        self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
    )
    attn_weights = torch.softmax(
        attn_scores / keys.shape[-1]**0.5, dim=-1
    )
    attn_weights = dropout(attn_weights)

    context_vec = attn_weights @ values
    return context_vec

In [ ]:
torch.manual_seed(123)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print(f"context_vecs.shape: {context_vecs.shape}")

context_vecs.shape: torch.Size([2, 6, 2])


#3.6 Extending single-head attention to multi-head attention

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()
    self.heads = nn.ModuleList(
        [CausalAttention(
            d_in,d_out,context_length,dropout,qkv_bias
        )
        for _ in range(num_heads)]
    )
  def forward(self,x):
    return torch.cat([head(x) for head in self.heads], dim=-1)

In [ ]:
torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3,2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0,num_heads=2)
context_vecs = mha(batch)

print(context_vecs)
print(f"context_vecs.shape: {context_vecs.shape}")

tensor([[[-0.6357, -0.1859,  0.2343,  0.1723],
         [-0.5118, -0.1684,  0.4834,  0.3947],
         [-0.7776, -0.2389,  0.5966,  0.4618],
         [-0.1479,  0.0725,  0.5227,  0.3902],
         [-0.5422, -0.1692,  0.4725,  0.3869],
         [-0.9130, -0.2881,  0.8947,  0.5938]],

        [[-0.5025, -0.1652,  0.2997,  0.1348],
         [-0.2952, -0.1008,  0.7681,  0.4786],
         [-0.8158, -0.1470,  0.3625,  0.2897],
         [-0.5271, -0.1706,  0.7440,  0.5439],
         [-0.6787, -0.0962,  0.5175,  0.3862],
         [-0.4257, -0.1539,  0.6311,  0.4446]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


# 3.6.2 Implementing multi-head attention with weight splits


In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__( self, d_in,d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()
    assert (d_out % num_heads == 0), \
         "d_out must be divisible by num_heads"

    self.d_out = d_out
    self.num_heads = num_heads
    self.head_dim = d_out // num_heads #Reduces the projection dim to match the desired output dim

    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    self.out_proj = nn.Linear(d_out,d_out) # uses a linear layer to combine head outputs

    self.dropout = nn.Dropout(dropout)
    self.register_buffer(
        'mask',
        torch.triu(torch.ones(context_length, context_length),
                   diagonal = 1)
    )

  def forward(self,x):
    b, num_tokens, d_in = x.shape

    keys = self.W_key(x) # Tensor shape: (b, num_tokens, d_out)
    queries = self.W_query(x) # Tensor shape: (b, num_tokens, d_out)
    values = self.W_value(x) # Tensor shape: (b, num_tokens, d_out)

    # We implicitly split the matrixby adding the num_heads dimension
    # Then we unroll the last dim: (b,num_tokens,dout) -> (b,num_tokens, num_heads, head_dims)
    keys = keys.view(b,num_tokens,self.num_heads,self.head_dim)
    queries = queries.view(b,num_tokens,self.num_heads,self.head_dim)
    values = values.view(b,num_tokens,self.num_heads,self.head_dim)

    #Transpose (b, num_tokens, num_heads, head_dims)-> (b, num_heads, num_tokens, head_dims)
    keys = keys.transpose(1,2)
    queries = queries.transpose(1,2)
    values = values.transpose(1,2)

    #compute dot products for each head
    attn_scores = queries @ keys.transpose(2,3)
    mask_bool = self.mask.bool()[:num_tokens, :num_tokens] # mask turncated to the number of tokens

    attn_scores.masked_fill_(mask_bool, -torch.inf) #used the mask to fill atention scores

    attn_weights = torch.softmax(
        attn_scores / keys.shape[-1] ** 0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)

    context_vec = (attn_weights @ values).transpose(1,2) # Tensor shape: (b,num_tokens,n_heads,head_dim)

    context_vec = context_vec.contiguous().view( #combines heads where self.dout = self.num_heads * self.head_dim
        b,num_tokens,self.d_out
    )
    context_vec = self.out_proj(context_vec) # adds an optional linear projection

    return context_vec


In [ ]:
#as an example

# (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

In [ ]:
print(a@a.transpose(2,3))


tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


In [ ]:
first_head = a[0,0,:,:]
first_res = first_head @ first_head.T
print(f"First Head:\n{first_res}")

second_head = a[0,1,:,:]
second_res = second_head @ second_head.T
print(f"Second Head:\n{second_res}")

First Head:
tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])
Second Head:
tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


In [ ]:
#using the MultiHeadAttention

torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print(f"context_vecs.shape:{context_vecs.shape}")

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape:torch.Size([2, 6, 2])
